# 01_run-single-match-prediction

World Cup 2026, Single match Monte Carlo simulation using Gradient Boosting based on FIFA ranking and historical features. This notebook outputs the following information:

- home-away win probabilities
- draw probability
- expected goals for each team
- most common exact score
- recommended score tip
- expected points for the recommended tip


In [ ]:
import numpy as np
import random
import matplotlib.pyplot as plt
import seaborn as sns

from utils.simulation import (
    find_optimal_tip_from_simulations,
    simulate_match_many,
)

RANDOM_SEED = 42
CURRENT_YEAR = 2026
DRAW_THRESHOLD = 0.30

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)


## Configuration

Adjust `team_*`, `phase`, and `n_simulations` depending on the use case.

Scoreline sampling now uses Poisson only.

In [ ]:
team_a = 'Argentina'
team_b = 'Katar'
phase = 'group' # one of 'group', 'play-in', 'round_of_16', 'quarterfinal', 'semifinal', 'final'
n_simulations = 10_000

## Run the match simulations

Run Monte Carlo simulation for matchup.

In [ ]:
df_simulations = simulate_match_many(
    team_a=team_a,
    team_b=team_b,
    phase=phase,
    n_simulations=n_simulations,
)

dict_summary = {
    f'{team_a}_win_percent': (df_simulations['result'] == 'A').mean() * 100,
    'draw_percent': (df_simulations['result'] == 'D').mean() * 100,
    f'{team_b}_win_percent': (df_simulations['result'] == 'B').mean() * 100,
    f'{team_a}_avg_goals': df_simulations['goals_a'].mean(),
    f'{team_b}_avg_goals': df_simulations['goals_b'].mean(),
    f'{team_a}_avg_gd': (
        df_simulations['goals_a'] - df_simulations['goals_b']
    ).mean(),
}

df_tiprank = find_optimal_tip_from_simulations(
    df_simulations=df_simulations,
    team_a=team_a,
    team_b=team_b,
    phase=phase,
    max_tip_goals=6,
)

## Plot simulation outcomes

In [ ]:
from matplotlib.patches import Patch

top_n = 15
n_simulations = len(df_simulations)

team_a_color = '#1f77b4'
team_b_color = '#d62728'
draw_color = '#8c8c8c'
highlight_color = '#2ca02c'
grid_color = '#e6e6e6'

df_topscores = (
    df_simulations['score']
    .value_counts(normalize=True)
    .head(top_n)
    .mul(100)
)

result_labels = {
    'A': f'{team_a} win',
    'D': 'Draw',
    'B': f'{team_b} win',
}

df_distributions = (
    df_simulations['result']
    .map(result_labels)
    .value_counts(normalize=True)
    .mul(100)
)

df_tiprank_top = df_tiprank.head(top_n).copy()

df_goal_diff = (
    (df_simulations['goals_a'] - df_simulations['goals_b'])
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
)

best_tip = df_tiprank.iloc[0]
most_likely_score = df_topscores.index[0]
most_likely_score_prob = df_topscores.iloc[0]
most_likely_outcome = df_distributions.index[0]
most_likely_outcome_prob = df_distributions.iloc[0]

def score_to_color(score):
    """Return colour based on the score outcome."""
    goals_a, goals_b = map(int, score.split('-'))

    if goals_a > goals_b:
        return team_a_color
    if goals_b > goals_a:
        return team_b_color
    return draw_color

def tip_to_color(row):
    """Return colour based on the tip outcome, highlighting the best tip."""
    if row.name == df_tiprank_top.index[0]:
        return highlight_color

    if row['tip_goals_a'] > row['tip_goals_b']:
        return team_a_color
    if row['tip_goals_b'] > row['tip_goals_a']:
        return team_b_color
    return draw_color

def goal_diff_to_color(goal_diff):
    """Return colour based on goal difference sign."""
    if goal_diff > 0:
        return team_a_color
    if goal_diff < 0:
        return team_b_color
    return draw_color

score_colors = [score_to_color(score) for score in df_topscores.index]
outcome_colors = [
    team_a_color if label == f'{team_a} win'
    else team_b_color if label == f'{team_b} win'
    else draw_color
    for label in df_distributions.index
]
tip_colors = df_tiprank_top.apply(tip_to_color, axis=1).tolist()
goal_diff_colors = [
    goal_diff_to_color(goal_diff)
    for goal_diff in df_goal_diff.index
]

df_scores_plot = df_topscores.rename_axis('score').reset_index(name='probability')
score_palette = dict(zip(df_scores_plot['score'], score_colors))

df_outcomes_plot = df_distributions.rename_axis('outcome').reset_index(name='probability')
outcome_palette = dict(zip(df_outcomes_plot['outcome'], outcome_colors))

df_tips_plot = df_tiprank_top.copy()
tip_palette = dict(zip(df_tips_plot['tip'], tip_colors))

df_goal_diff_plot = df_goal_diff.rename_axis('goal_diff').reset_index(name='probability')
goal_diff_palette = dict(zip(df_goal_diff_plot['goal_diff'], goal_diff_colors))

plt.rcParams.update({
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.titleweight': 'bold',
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
})

fig, axes = plt.subplots(2, 2, figsize=(18, 12))
fig.patch.set_facecolor('white')

fig.suptitle(
    f'Simulation Summary - {team_a} vs {team_b}\n'
    f'(phase={phase}, n={n_simulations:,})',
    fontsize=19,
    # fontweight='bold',
    y=0.99,
)

# Most common scores.
sns.barplot(
    data=df_scores_plot,
    x='probability',
    y='score',
    hue='score',
    palette=score_palette,
    legend=False,
    ax=axes[0, 0],
)
axes[0, 0].set_title('Most Common Scores')
axes[0, 0].set_xlabel('Probability (%)')
axes[0, 0].set_ylabel('Score')
axes[0, 0].grid(axis='x', color=grid_color, linewidth=1)
axes[0, 0].set_axisbelow(True)

for container in axes[0, 0].containers:
    axes[0, 0].bar_label(container, fmt='%.1f%%', padding=3)

# Outcome distribution.
sns.barplot(
    data=df_outcomes_plot,
    x='probability',
    y='outcome',
    hue='outcome',
    palette=outcome_palette,
    legend=False,
    ax=axes[0, 1],
)
axes[0, 1].set_title('Outcome Distribution')
axes[0, 1].set_xlabel('Probability (%)')
axes[0, 1].set_ylabel('')
axes[0, 1].grid(axis='x', color=grid_color, linewidth=1)
axes[0, 1].set_axisbelow(True)

for container in axes[0, 1].containers:
    axes[0, 1].bar_label(container, fmt='%.1f%%', padding=3)

# Best tips.
sns.barplot(
    data=df_tips_plot,
    x='expected_points',
    y='tip',
    hue='tip',
    palette=tip_palette,
    legend=False,
    ax=axes[1, 0],
)
axes[1, 0].set_title('Best Tips by Expected Points')
axes[1, 0].set_xlabel('Expected Points')
axes[1, 0].set_ylabel('Tip')
axes[1, 0].grid(axis='x', color=grid_color, linewidth=1)
axes[1, 0].set_axisbelow(True)

for container in axes[1, 0].containers:
    axes[1, 0].bar_label(container, fmt='%.2f', padding=3)

# Goal-difference distribution.
sns.barplot(
    data=df_goal_diff_plot,
    x='goal_diff',
    y='probability',
    hue='goal_diff',
    palette=goal_diff_palette,
    legend=False,
    ax=axes[1, 1],
)
axes[1, 1].set_title('Goal-Difference Distribution')
axes[1, 1].set_xlabel(f'Goal Difference ({team_a} - {team_b})')
axes[1, 1].set_ylabel('Probability (%)')
axes[1, 1].grid(axis='y', color=grid_color, linewidth=1)
axes[1, 1].set_axisbelow(True)

for container in axes[1, 1].containers:
    axes[1, 1].bar_label(container, fmt='%.1f%%', padding=3)

# Give labels some breathing room.
for ax in axes.flat:
    ax.margins(x=0.08)
    ax.tick_params(axis='both', length=0)

# Legend.
legend_handles = [
    Patch(color=team_a_color, label=f'{team_a} win / positive GD'),
    Patch(color=draw_color, label='Draw / GD 0'),
    Patch(color=team_b_color, label=f'{team_b} win / negative GD'),
    Patch(color=highlight_color, label='Recommended tip'),
]

fig.legend(
    handles=legend_handles,
    loc='lower center',
    ncol=4,
    frameon=False,
    fontsize=11,
    bbox_to_anchor=(0.5, -0.01),
)

plt.tight_layout(rect=[0, 0.04, 1, 0.95])
plt.show()

print(f'Recommended tip: {team_a} {best_tip["tip"]} {team_b}')
print(f'Expected points: {best_tip["expected_points"]:.2f}')
print(
    'Exact score probability: '
    f'{best_tip["exact_score_probability_percent"]:.2f}%'
)
print(f'Tip outcome: {best_tip["outcome"]}')
print()
print(
    f'Most likely exact score: '
    f'{team_a} {most_likely_score} {team_b} '
    f'({most_likely_score_prob:.1f}%)'
)
print(
    f'Most likely outcome: '
    f'{most_likely_outcome} ({most_likely_outcome_prob:.1f}%)'
)